In [209]:
import pandas as pd 
import numpy as np
import re
from sklearn.metrics import accuracy_score, f1_score

In [210]:
#file reading function
def extract_text(path):
    with open(path, 'r', encoding = 'utf-8') as f:
        text = f.read().splitlines()
    return text

train_text = extract_text('train_text.txt')
val_text = extract_text('val_text.txt')
test_text = extract_text('test_text.txt')

In [211]:
#dataframe for training set
train_labels = pd.read_csv('train_labels.txt', header=None, names=['label'])
traindf = pd.DataFrame({'text': train_text, 'label': train_labels['label']})

#dataframe for validation set
val_labels = pd.read_csv('val_labels.txt', header=None, names=['label'])
valdf = pd.DataFrame({'text': val_text, 'label': val_labels['label']})

#dataframe for test set
test_labels = pd.read_csv('test_labels.txt', header=None, names=['label'])
testdf = pd.DataFrame({'text': test_text, 'label': test_labels['label']})


In [212]:
#lexicon reading
lex_adj = pd.read_csv("socialsent_hist_adj/adjectives/2000.tsv", sep="\t", names = ["word", "score"])
lex_freq = pd.read_csv('socialsent_hist_freq/frequent_words/2000.tsv', sep="\t", names = ["word", "score"])

In [213]:
import glob
import os

In [214]:
#lexicon reading for subreddits
subr = glob.glob("socialsent_subreddits/subreddits/*.tsv")
print(subr)
subr_lex = []
seen_chars = set()
for s in subr:
    f_name = os.path.basename(s)
    f_char = f_name[0]
    if f_char.isdigit():
        df = pd.read_csv(s,sep="\t", names = ["word", "score"])
        subr_lex.append(df)
        seen_chars.add(f_char)

    elif f_char.isalpha() and f_char not in seen_chars:
        df = pd.read_csv(s, sep = "\t", names = ["word", "score"])
        subr_lex.append(df)
        seen_chars.add(f_char)
        
subr_lex = pd.concat(subr_lex)

['socialsent_subreddits/subreddits\\2007scape.tsv', 'socialsent_subreddits/subreddits\\3DS.tsv', 'socialsent_subreddits/subreddits\\4chan.tsv', 'socialsent_subreddits/subreddits\\ACTrade.tsv', 'socialsent_subreddits/subreddits\\AdviceAnimals.tsv', 'socialsent_subreddits/subreddits\\amiugly.tsv', 'socialsent_subreddits/subreddits\\Anarcho_Capitalism.tsv', 'socialsent_subreddits/subreddits\\Android.tsv', 'socialsent_subreddits/subreddits\\anime.tsv', 'socialsent_subreddits/subreddits\\apple.tsv', 'socialsent_subreddits/subreddits\\archeage.tsv', 'socialsent_subreddits/subreddits\\AskMen.tsv', 'socialsent_subreddits/subreddits\\askscience.tsv', 'socialsent_subreddits/subreddits\\AskWomen.tsv', 'socialsent_subreddits/subreddits\\asoiaf.tsv', 'socialsent_subreddits/subreddits\\atheism.tsv', 'socialsent_subreddits/subreddits\\australia.tsv', 'socialsent_subreddits/subreddits\\aww.tsv', 'socialsent_subreddits/subreddits\\BabyBumps.tsv', 'socialsent_subreddits/subreddits\\baseball.tsv', 'socia

In [215]:
print(subr_lex)

           word  score
pisses    -3.92   1.22
blizzard  -3.75   1.74
fucked    -3.75   1.52
complains -3.74   2.10
garbage   -3.67   1.71
...         ...    ...
mat        3.62   1.26
artwork    3.66   1.12
art        3.71   1.18
sexy       3.81   1.08
beautiful  3.82   0.69

[219438 rows x 2 columns]


In [216]:
#create dictionaries for lexicons
adj_dict = dict(zip(lex_adj['word'], lex_adj['score']))
freq_dict = dict(zip(lex_freq['word'], lex_freq['score']))
subr_dict = dict(zip(subr_lex.word, subr_lex.score))

In [217]:
#tokenization function
def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = text.split()
    return tokens

In [218]:
#feature extraction function
def extract_features(text):

    words = tokenize(text)

    # --- Lexicon Features ---
    lex_scores_adj = []
    lex_scores_freq = []
    lex_scores_sub = []

    for w in words:
        if w in adj_dict:
            lex_scores_adj.append(adj_dict[w])

        if w in freq_dict:
            lex_scores_freq.append(freq_dict[w])

        if w in subr_dict:
            lex_scores_sub.append(subr_dict[w])

    # Mean sentiment scores
    f1 = np.mean(lex_scores_adj) if lex_scores_adj else 0
    f2 = np.mean(lex_scores_freq) if lex_scores_freq else 0
    f3 = np.mean(lex_scores_sub) if lex_scores_sub else 0

    # lexicon-style features
    f4 = len([x for x in lex_scores_adj if x > 0])
    f5 = len([x for x in lex_scores_adj if x < 0])

    f6 = len([x for x in lex_scores_freq if x > 0])
    f7 = len([x for x in lex_scores_freq if x < 0])

    f8 = len([x for x in lex_scores_sub if x > 0])
    f9 = len([x for x in lex_scores_sub if x < 0])

    #Text Statistics
    word_count = len(words)
    longest_word = max([len(w) for w in words]) if words else 1
    long_words = len([w for w in words if len(w) >= 5])

    f10 = np.log(word_count + 1)
    f11 = np.log(longest_word + 1)
    f12 = np.log(long_words + 1)

    return [f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12]

In [219]:
X_train = np.array([extract_features(t) for t in train_text])
X_val = np.array([extract_features(t) for t in val_text])
X_test = np.array([extract_features(t) for t in test_text])

y_train = train_labels.values
y_test = test_labels.values


In [220]:
X_train = np.hstack([np.ones((X_train.shape[0],1)), X_train])
X_test = np.hstack([np.ones((X_test.shape[0],1)), X_test])


In [221]:
def sigmoid(z):
    return 1/(1+np.exp(-z))

In [222]:
def train_logistic(X, y, lr=0.01, epochs=1000):

    w = np.zeros(X.shape[1])

    for _ in range(epochs):
        z = np.dot(X, w)
        preds = sigmoid(z)

        gradient = np.dot(X.T, (preds - y)) / len(y)
        w -= lr * gradient

    return w

In [223]:
def predict(X, w):
    return (sigmoid(np.dot(X,w)) > 0.5).astype(int)

In [224]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

model.fit(X_train, y_train)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)


print("Train Accuracy:", accuracy_score(y_train, train_pred))
print("Test Accuracy:", accuracy_score(y_test, test_pred))
print("Test F1:", f1_score(y_test, test_pred, average="macro"))

c:\Users\rapst\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Train Accuracy: 0.4533157952427929
Test Accuracy: 0.47451970042331487
Test F1: 0.2503411071555555
